# Random Forest Regression for Short-Horizon Ship Motion Prediction

This notebook:
1. Loads the Maxsurf time-series Excel file (uploaded in Colab).
2. Uses the **"Time-Series Data"** sheet with 6-DoF motions.
3. Resamples the data to **10 Hz** using linear interpolation.
4. Builds supervised learning windows:
   - Input: last **100 timesteps (10 s)** of 6-DoF motions → 100×6 = 600 features.
   - Output: next **50 timesteps (5 s)** of 6-DoF motions → 50×6 = 300 targets.
5. Performs chronological **80/20 train–test split** (no shuffling).
6. Trains a **RandomForestRegressor** multi-output model.
7. Evaluates using **overall RMSE/MAE** and **per-motion RMSE**.

In [ ]:
# Cell 1: Imports

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# Cell 2: Load Excel file and select sheet

# By default, we look for "BTP.xlsx".
# If not found, we automatically pick the first .xlsx file in the current directory.
default_excel_path = "BTP.xlsx"

if os.path.exists(default_excel_path):
    excel_path = default_excel_path
else:
    xlsx_files = [f for f in os.listdir() if f.lower().endswith(".xlsx")]
    if not xlsx_files:
        raise FileNotFoundError(
            "No .xlsx file found. Please upload your Excel file (e.g., BTP.xlsx) "
            "to the Colab working directory and re-run this cell."
        )
    excel_path = xlsx_files[0]

print("Using Excel file:", excel_path)

# Sheet that contains time-series with 6-DoF
sheet_name = "Time-Series Data"

# Read the sheet
df = pd.read_excel(excel_path, sheet_name=sheet_name)

print("Loaded sheet:", sheet_name)
print("DataFrame shape:", df.shape)
print("Columns:", list(df.columns))

Using Excel file: BTP.xlsx
Loaded sheet: Time-Series Data
DataFrame shape: (1272, 8)
Columns: ['time series', 'wave[m]', 'surge[m]', 'sway[m]', 'heave[m]', 'roll[deg]', 'pitch[deg]', 'yaw[deg]']


In [ ]:
# Cell 3: Basic cleaning and ordering by time

# Expected column names (case-sensitive, as in the provided file)
time_col = "time series"
wave_col = "wave[m]"
motion_cols = ["surge[m]", "sway[m]", "heave[m]", "roll[deg]", "pitch[deg]", "yaw[deg]"]

# Ensure required columns exist
required_cols = [time_col, wave_col] + motion_cols
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in sheet '{sheet_name}': {missing}")

# Sort by time (in case it's not already sorted)
df = df.sort_values(by=time_col).reset_index(drop=True)

print("Sorted by time. Time range:",
      float(df[time_col].iloc[0]), "to", float(df[time_col].iloc[-1]))

Sorted by time. Time range: 0.0 to 232.864


In [ ]:
# Cell 4: Resample to 10 Hz (dt = 0.1 s) using linear interpolation

# Original time and data
orig_t = df[time_col].values.astype(float)

# New regular time grid at 10 Hz
dt = 0.1  # seconds
t_start = orig_t[0]
t_end = orig_t[-1]

# Ensure inclusive of the final time (numerical stability)
new_t = np.arange(t_start, t_end + dt * 0.5, dt)

resampled_data = {time_col: new_t}

# Interpolate wave and 6-DoF motions
for col in [wave_col] + motion_cols:
    resampled_data[col] = np.interp(new_t, orig_t, df[col].values.astype(float))

df_resampled = pd.DataFrame(resampled_data)

print("Resampled data shape:", df_resampled.shape)
print("Approximate sampling frequency: {:.2f} Hz".format(1.0 / dt))

Resampled data shape: (2330, 8)
Approximate sampling frequency: 10.00 Hz


In [ ]:
# Cell 5: Build supervised learning windows (X, y)

# We use only the 6-DoF motions as features/targets
data = df_resampled[motion_cols].values  # shape: [T, 6]

input_length = 100   # timesteps (10 s)
output_length = 50   # timesteps (5 s)
n_dofs = data.shape[1]

total_len = data.shape[0]
window_size = input_length + output_length

if total_len <= window_size:
    raise ValueError(
        f"Not enough time steps after resampling (T={total_len}) "
        f"for window_size={window_size} (input+output)."
    )

# Number of samples with sliding window (stride = 1)
n_samples = total_len - window_size + 1

X = np.zeros((n_samples, input_length * n_dofs), dtype=np.float32)
y = np.zeros((n_samples, output_length * n_dofs), dtype=np.float32)

for i in range(n_samples):
    past = data[i : i + input_length]                     # shape: [100, 6]
    future = data[i + input_length : i + window_size]     # shape: [50, 6]
    X[i] = past.reshape(-1)
    y[i] = future.reshape(-1)

print("Total time steps:", total_len)
print("Number of samples (windows):", n_samples)
print("X shape (features):", X.shape)   # [N_samples, 600]
print("y shape (targets):", y.shape)    # [N_samples, 300]

Total time steps: 2330
Number of samples (windows): 2181
X shape (features): (2181, 600)
y shape (targets): (2181, 300)


In [ ]:
# Cell 6: Chronological 80/20 train-test split (no shuffling)

train_ratio = 0.8
split_idx = int(n_samples * train_ratio)

X_train = X[:split_idx]
y_train = y[:split_idx]
X_test = X[split_idx:]
y_test = y[split_idx:]

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Train samples: 1744
Test samples: 437


In [ ]:
# Cell 7: Define and train the Random Forest regression model (multi-output)

rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=16,
    min_samples_split=3,
    min_samples_leaf=2,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

print("Training RandomForestRegressor...")
rf.fit(X_train, y_train)
print("Training complete.")

Training RandomForestRegressor...
Training complete.


In [ ]:
# Cell 8: Evaluation - overall RMSE & MAE, plus per-motion RMSE

# Predict on test set
y_pred = rf.predict(X_test)

# Overall metrics across all outputs (flattened 300-dimensional vectors)
overall_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
overall_mae = mean_absolute_error(y_test, y_pred)

print("=== Overall Performance (all 300 outputs) ===")
print(f"RMSE: {overall_rmse:.6f}")
print(f"MAE : {overall_mae:.6f}\n")

# Per-motion RMSE
n_test = y_test.shape[0]
# Reshape to [N_test, output_length, n_dofs]
y_test_seq = y_test.reshape(n_test, output_length, n_dofs)
y_pred_seq = y_pred.reshape(n_test, output_length, n_dofs)

print("=== Per-motion RMSE (aggregated over test samples & future timesteps) ===")
for j, name in enumerate(motion_cols):
    true_flat = y_test_seq[:, :, j].ravel()
    pred_flat = y_pred_seq[:, :, j].ravel()
    rmse_j = np.sqrt(mean_squared_error(true_flat, pred_flat))
    print(f"{name:10s} RMSE: {rmse_j:.6f}")

=== Overall Performance (all 300 outputs) ===
RMSE: 0.000956
MAE : 0.000582

=== Per-motion RMSE (aggregated over test samples & future timesteps) ===
surge[m]   RMSE: 0.000080
sway[m]    RMSE: 0.000194
heave[m]   RMSE: 0.000797
roll[deg]  RMSE: 0.001169
pitch[deg] RMSE: 0.001824
yaw[deg]   RMSE: 0.000338
